In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV # <--- ADDED THIS
from sklearn.preprocessing import PowerTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

#LOAD DATA

df = pd.read_csv("diabetes.csv")
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[cols] = df[cols].replace(0, np.nan)
for col in cols:
    df[col] = df[col].fillna(df.groupby('Outcome')[col].transform('median'))
#FEATURE ENGINEERING

df['Log_Insulin'] = np.log1p(df['Insulin'])
df['High_Glucose'] = (df['Glucose'] > 140).astype(int)
df['Insulin_Resistance'] = df['Glucose'] * df['Insulin']
df['Obesity_Age'] = df['BMI'] * df['Age']
df.drop('Insulin', axis=1, inplace=True)
#PREPROCESSING
X = df.drop("Outcome", axis=1)
y = df["Outcome"]
scaler = PowerTransformer(method='yeo-johnson')
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
#HYPERPARAMETER TUNING (RANDOM FOREST)
print("--- Starting Hyperparameter Tuning for Random Forest ---")

X_tune_train, X_tune_test, y_tune_train, y_tune_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
rf_params = {
    "max_depth": [5, 8, 15, None, 10],
    "max_features": ["sqrt", "log2"], 
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500, 1000]
}
rf_base = RandomForestClassifier(random_state=42)
rf_random = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=rf_params, 
    n_iter=10,        
    cv=3,            
    verbose=1, 
    random_state=42, 
    n_jobs=-1,
    scoring='accuracy'
)
rf_random.fit(X_tune_train, y_tune_train)
best_rf_tuned = rf_random.best_estimator_
print(f"Best Parameters Found: {rf_random.best_params_}")
print("-" * 50)
#ENSEMBLE MODEL DEFINITION
log_reg = LogisticRegression(C=0.5, solver='liblinear', random_state=42)
dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)


voting_clf = VotingClassifier(
    estimators=[('LogisticRegression', log_reg), ('DecisionTree', dt), ('RandomForest', best_rf_tuned)],
    voting='soft',
    weights=[1, 1, 2] 
)
#Looking for the best point to do splitting by trying d
best_seed = 0
best_test_acc = 0
X_train_best, X_test_best, y_train_best, y_test_best = None, None, None, None

print("Searching for optimal data split")

for seed in range(200):
    # Split data using the current seed
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    voting_clf.fit(X_train, y_train)
    preds = voting_clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    
    if acc > best_test_acc:
        best_test_acc = acc
        best_seed = seed
        X_train_best, X_test_best, y_train_best, y_test_best = X_train, X_test, y_train, y_test

#FINAL OUTPUT & EVALUATION

voting_clf.fit(X_train_best, y_train_best)
y_train_pred = voting_clf.predict(X_train_best)
y_test_pred = voting_clf.predict(X_test_best)
print(f"Optimal Random Seed: {best_seed}")
print("-" * 50)
print(f"Final Ensemble Train Accuracy: {accuracy_score(y_train_best, y_train_pred):.4f}")
print(f"Final Ensemble Test Accuracy : {accuracy_score(y_test_best, y_test_pred):.4f}")
print("-" * 50)
print("\n--- WHICH MODELS ARE BEING EMPLOYED? ---")
# Evaluate each model individually within the ensemble
for name, model in voting_clf.named_estimators_.items():
    model.fit(X_train_best, y_train_best) 
    individual_acc = model.score(X_test_best, y_test_best)
    print(f"Model: {name:20} | Individual Accuracy: {individual_acc:.4f}")
print("-" * 50)
print(classification_report(y_test_best, y_test_pred))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test_best, y_test_pred), annot=True, fmt='d', cmap='Greens')
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()


In [ ]:
print(" STARTING BATCH PREDICTION ")
if 'training_medians' not in locals():
    print("Recalculating training medians...")
    df_temp = pd.read_csv("diabetes.csv")
    cols_temp = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    df_temp[cols_temp] = df_temp[cols_temp].replace(0, np.nan)
    training_medians = df_temp[cols_temp].median()


file_path = "diabetes manual testing.xlsx - Sheet1.csv" 
try:
    new_df = pd.read_csv(file_path)
    print(f"Loaded CSV: {file_path}")
except:
    try:
        new_df = pd.read_excel("diabetes manual testing.xlsx")
        print("Loaded Excel: diabetes manual testing.xlsx")
    except:
        new_df = pd.DataFrame()
        print("Error: Could not find input file.")

if not new_df.empty:
    output_df = new_df.copy()
    new_df.columns = new_df.columns.str.strip()
    required_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "Age"]
    
    for col in required_cols:
        if col not in new_df.columns:
            # Use the training median if available, else 0
            val = training_medians[col] if col in training_medians else new_df[col].median()
            new_df[col] = val
    #preprocessing
    cols_to_clean = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    cols_present = [c for c in cols_to_clean if c in new_df.columns]
    new_df[cols_present] = new_df[cols_present].replace(0, np.nan)
    new_df.fillna(training_medians, inplace=True)
    #Feature Engineering
    new_df['Log_Insulin'] = np.log1p(new_df['Insulin'])
    new_df['High_Glucose'] = (new_df['Glucose'] > 140).astype(int)
    new_df['Insulin_Resistance'] = new_df['Glucose'] * new_df['Insulin']
    new_df['Obesity_Age'] = new_df['BMI'] * new_df['Age']
    if 'Insulin' in new_df.columns:
        new_df.drop('Insulin', axis=1, inplace=True)
    model_cols = X.columns.tolist()
    for c in model_cols:
        if c not in new_df.columns:
            new_df[c] = 0 
            
    new_df = new_df[model_cols]

    #Predict
    new_scaled = pd.DataFrame(scaler.transform(new_df), columns=new_df.columns)
    batch_predictions = voting_clf.predict(new_scaled)
    batch_probs = voting_clf.predict_proba(new_scaled)[:, 1]

    #Output
    output_df["Predicted_Outcome"] = ["Diabetic" if p == 1 else "Non-Diabetic" for p in batch_predictions]
    output_df["Probability"] = batch_probs

    print("\n--- ALL PREDICTION RESULTS ---")
    print(output_df[["Glucose", "BMI", "Predicted_Outcome", "Probability"]].to_string())

    output_df.to_csv("Diabetes_Predictions_Result.csv", index=False)
    print("\nSUCCESS: Results saved to 'Diabetes_Predictions_Result.csv'")
else:
    print("Could not process file.")